# 04 — Classify with DarkBERT

**Goal.** Apply the DarkBERT classifier you fine-tuned in `cti_301/04_darkbert_coda.ipynb` to the corpus from notebook 03, and write the predicted `label` + `score` back into the `documents` table.

**No training here.** Inference only. fp16 + batch 16 fits comfortably in 8 GB VRAM.

**Inputs.**
- `db/cti.duckdb` (from nb 03; or after nb 05 if you want Telegram included)
- `../cti_301/models/darkbert_coda/` — adjust the path below if yours differs.

**Output.** Same DuckDB, with `label` and `score` populated for one document per `dedup_group` (we don't waste GPU on duplicates).

In [3]:
from pathlib import Path
import duckdb, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = Path("../cti_301/models/darkbert_coda_final")
DB_PATH = Path("db/cti.duckdb")

assert MODEL_DIR.exists(), f"Set MODEL_DIR to your cti_301 DarkBERT checkpoint. Not found: {MODEL_DIR}"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


## 1. Load model + tokenizer

fp16 halves VRAM at negligible accuracy cost for inference. If you're on CPU, drop the `.half()`.

In [4]:
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval().to(device)
if device == "cuda":
    model = model.half()

id2label = model.config.id2label
print("labels:", id2label)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

labels: {0: 'Arms', 1: 'Crypto', 2: 'Drugs', 3: 'Electronic', 4: 'Financial', 5: 'Gambling', 6: 'Hacking', 7: 'Porn', 8: 'Violence'}


## 2. Pull one document per dedup group

Within a dedup cluster all texts are near-identical — we classify the leader and propagate the label to siblings at the end.

In [7]:
con = duckdb.connect(str(DB_PATH))
rows = con.execute("""
    WITH leaders AS (
        SELECT dedup_group, MIN(url_id) AS leader_id
        FROM documents
        GROUP BY dedup_group
    )
    SELECT d.url_id, d.text
    FROM documents d
    JOIN leaders l ON d.url_id = l.leader_id
    WHERE d.label IS NULL
""").fetchall()
print(f"{len(rows)} dedup leaders to classify")

4 dedup leaders to classify


## 3. Batched fp16 inference

In [8]:
BATCH = 16
MAX_LEN = 512  # DarkBERT's RoBERTa-base context

@torch.inference_mode()
def predict_batch(texts):
    enc = tok(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(device)
    logits = model(**enc).logits
    probs = logits.softmax(-1)
    scores, ids = probs.max(-1)
    return [id2label[i.item()] for i in ids], scores.float().cpu().tolist()

preds = []
for i in range(0, len(rows), BATCH):
    chunk = rows[i:i+BATCH]
    labels, scores = predict_batch([r[1] for r in chunk])
    for (uid, _), lbl, sc in zip(chunk, labels, scores):
        preds.append((lbl, sc, uid))
    if (i // BATCH) % 5 == 0:
        print(f"  {i + len(chunk)}/{len(rows)}")
print(f"done: {len(preds)} predictions")

  4/4
done: 4 predictions


## 4. Write back to DuckDB

First the leaders, then propagate to their dedup-group siblings in a single SQL update.

In [9]:
con.executemany("UPDATE documents SET label = ?, score = ? WHERE url_id = ?", preds)

con.execute("""
    UPDATE documents AS d
    SET label = leader.label, score = leader.score
    FROM (
        SELECT dedup_group, ANY_VALUE(label) AS label, ANY_VALUE(score) AS score
        FROM documents
        WHERE label IS NOT NULL
        GROUP BY dedup_group
    ) AS leader
    WHERE d.dedup_group = leader.dedup_group AND d.label IS NULL
""")

print("label coverage:")
print(con.execute("SELECT label, COUNT(*) FROM documents GROUP BY label ORDER BY 2 DESC").fetchdf())

label coverage:
      label  count_star()
0    Crypto             2
1  Violence             1
2   Hacking             1


## 5. Spot-check the highest-confidence hits per label

In [10]:
con.execute("""
    SELECT label, score, title, SUBSTR(text, 1, 160) AS preview
    FROM documents
    WHERE label IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (PARTITION BY label ORDER BY score DESC) <= 2
    ORDER BY label, score DESC
""").fetchdf()

,label,score,title,preview
0,Crypto,0.966797,The Tor Project | Privacy & Freedom Online,Tor Browser isolates each website you visit so...
1,Crypto,0.890625,Ahmia,Ahmia\nAhmia searches hidden services on the T...
2,Hacking,0.963867,Share and accept documents securely,Latest News\nSecureDrop Inbox 1.0.0 releasedSh...
3,Violence,0.984375,BBC - Home,BBC Homepage\nLive.Â\nPolice declare terrorist...


## What's next

Notebook 05 ingests recent messages from a few public CTI Telegram channels into the same `documents` table (`source='telegram'`). You can re-run this notebook afterwards to classify them too — it skips rows that already have a label.